# 2 SQL DATABASE

**Project:** Irish Milk Supply: Growth, Seasonality and Forecasting

**Author:** Temuulen Bulgan

### Purpose of this notebook
This notebook loads the clean data from notebook 1 into the SQL Server database `dairy`. The database and its tables were created with the scripts in the `sql/` folder.

### Input
`data/processed/ireland_dairy_monthly.csv` and `data/processed/canada_milk_monthly.csv`

### Output
The three tables in the `dairy` database are filled: `dim_date`, `dim_measure` and `fact_monthly`.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
import pyodbc
from sqlalchemy import create_engine

#### Check the ODBC driver

Python needs an ODBC driver to connect to SQL Server. I check which drivers are installed on this computer.

In [2]:
# List the ODBC drivers installed on this computer
pyodbc.drivers()

['SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)',
 'ODBC Driver 17 for SQL Server',
 'ODBC Driver 18 for SQL Server']

#### Connect to the database

I connect to the local SQL Server with Windows Authentication, the same way as in SSMS.

In [3]:
# Connection to the dairy database on this computer
engine = create_engine(
    "mssql+pyodbc://@localhost/dairy"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes",
    fast_executemany=True,
)

# Test the connection: list the tables in the dairy database
pd.read_sql("SELECT name FROM sys.tables ORDER BY name", engine)

,name
0,dim_date
1,dim_measure
2,fact_monthly


## 1. Load the clean data

In [4]:
# Load the clean tables from notebook 1, with the date as the row label
ireland = pd.read_csv("../data/processed/ireland_dairy_monthly.csv", index_col="date", parse_dates=True)
canada = pd.read_csv("../data/processed/canada_milk_monthly.csv", index_col="date", parse_dates=True)

print(ireland.shape, canada.shape)

(620, 7) (606, 1)


## 2. Build `dim_date`

One row per month. The Irish data covers January 1975 to August 2026, which includes every Canadian month, so I use the Irish dates.

The `quota_period` column labels each month by the EU milk quota regime: quotas started in April 1984 and ended on 31 March 2015.

In [5]:
# One row per month, with labels taken from the date
dates = ireland.index

dim_date = pd.DataFrame({
    "date_key": dates,
    "year": dates.year,
    "month": dates.month,
    "month_name": dates.month_name(),
    "quarter": dates.quarter,
})

# Quota period: quota by default, then pre-quota and post-quota by date
dim_date["quota_period"] = "quota"
dim_date.loc[dim_date["date_key"] < "1984-04-01", "quota_period"] = "pre-quota"
dim_date.loc[dim_date["date_key"] >= "2015-04-01", "quota_period"] = "post-quota"

dim_date.head(3)

,date_key,year,month,month_name,quarter,quota_period
0,1975-01-01,1975,1,January,1,pre-quota
1,1975-02-01,1975,2,February,1,pre-quota
2,1975-03-01,1975,3,March,1,pre-quota


In [6]:
# Check: number of months in each quota period, and the dates where each period starts and ends
dim_date.groupby("quota_period")["date_key"].agg(["count", "min", "max"])

,count,min,max
quota_period,,,
post-quota,137,2015-04-01,2026-08-01
pre-quota,111,1975-01-01,1984-03-01
quota,372,1984-04-01,2015-03-01


`dim_date` has 620 months: 111 before quotas (January 1975 to March 1984), 372 under quotas (April 1984 to March 2015) and 137 after quotas (April 2015 to August 2026).